# Verne, small, step 05: square up to the cube (a data step), on T1-wide

No explained notebook: the model is step 04's, unchanged; only the demos change. Step 04: `04_sampler.ipynb`.

Why: step 04's failures miss sideways, and the fingers close along y (the top camera's left-right). Fully open they leave (8.0 − 4.3) / 2 ≈ 1.8 cm of room each side of a square cube, but the expert never turned the wrist, and a diagonal cube is up to 6.1 cm wide across the fingers: ≈ 1.0 cm of room. Failures had more diagonal cubes (26° against 19° off square).

The expert (`expert.py`, `record.py`), four datasets of 100 kept demos, same T1-wide spawns and seeds:

```-
a  square_up        turn the wrist to the nearest face (≤ 45°) for the grasp, straighten after the lift
b  + full shake     joint noise σ 0.05 on the executed actions, clean labels (--noise-sigma)
c  + aim bias       per-episode wrist bias in ±45° executed until the lift, clean labels (--aim-bias-deg):
                    the gripper really closes crooked, every label still turns back to square
d  + both           bias and shake
```

Policy: step 04's package (VAE latents, 4 shifted sampler steps), training seed 0 per arm (`run job 05_square <task> 0`, local RTX 4060), rollouts on the plain T1-wide scene: the policy has to turn the wrist itself. Baseline: step 04, same seed, fixed-wrist demos.

In [ ]:
import json

from dream_robot.policies.verne.data import REPO
from dream_robot.policies.verne.run import RUNS, TASKS

ARMS = {
    'a: square': 't1-wide-square',
    'b: + shake': 't1-wide-square-noise',
    'c: + bias': 't1-wide-square-bias',
    'd: + bias + shake': 't1-wide-square-bias-noise',
}

### step 05.1 — the demos

The recorder keeps only placed episodes; perturbed experts fail some attempts, so their kept demos lean toward the cubes they could still place.

In [ ]:
for arm, task in ARMS.items():
    s = json.loads(
        (REPO / 'data/robosuite' / TASKS[task][0] / 'recording_summary.json').read_text()
    )
    print(
        f'{arm:18s} {s["kept"]} kept of {s["attempted"]} attempts ({s["expert_success_rate"]:.0%}), '
        f'{s["total_frames"]} frames, failures {s["failure_histogram"]}, perturbation {s["perturbation"]}'
    )

a: square          100 kept of 100 attempts (100%), 23436 frames, failures {}, perturbation None
b: + shake         100 kept of 118 attempts (85%), 25726 frames, failures {'dropped': 1, 'no_grasp': 17}, perturbation {'type': 'joint_noise', 'sigma': 0.05}
c: + bias          100 kept of 100 attempts (100%), 23767 frames, failures {}, perturbation {'type': 'aim_bias', 'degrees': 45.0}
d: + bias + shake  100 kept of 138 attempts (72%), 26464 frames, failures {'dropped': 4, 'no_grasp': 32, 'wrong_target': 2}, perturbation {'type': 'aim_bias', 'degrees': 45.0, 'noise': {'type': 'joint_noise', 'sigma': 0.05}}


Expert checks (`experiments/verne/scripts/expert_square.py`, 100 T1-wide spawns): the fixed-wrist expert closes 21.3° off square on average (worst 44.9°), the squaring one 1.9° (worst 7.8°); both place 100/100.

### step 05.2 — seed 0 of each arm, 50 rollout seeds, against step 04's seed 0

In [ ]:
def read_run(path):
    # a run folder -> n_exec -> (successes, failure counts), probe fit
    out = {}
    for n in (8, 16):
        res = json.loads((path / f'sweep_{n}.json').read_text()).values()
        fails = {}
        for ok, mode, _ in res:
            if not ok:
                fails[mode] = fails.get(mode, 0) + 1
        out[n] = (sum(r[0] for r in res), fails)
    return out, json.loads((path / 'probe.json').read_text())['fit']


rows = {'step 04, fixed wrist': RUNS / '04_sampler/s4_shift1.5/t1-wide/seed0'}
rows |= {arm: RUNS / '05_square' / task / 'seed0' for arm, task in ARMS.items()}

In [ ]:
print(
    f'{"":22s} {"n_exec 8":>9s} {"n_exec 16":>10s} {"miss":>7s} {"x / y slope":>13s}   failures (n_exec 8)'
)
for name, path in rows.items():
    r, fit = read_run(path)
    print(
        f'{name:22s} {r[8][0]:6d}/50 {r[16][0]:7d}/50 {fit["miss_cm"]:5.1f} cm '
        f'{fit["slope_x"]:+.2f} / {fit["slope_y"]:+.2f}   {r[8][1]}'
    )

                        n_exec 8  n_exec 16    miss   x / y slope   failures (n_exec 8)
step 04, fixed wrist       36/50      39/50   1.4 cm +1.03 / +0.98   {'no_grasp': 13, 'dropped': 1}
a: square                  28/50      25/50   1.5 cm +1.04 / +0.78   {'no_grasp': 18, 'wrong_target': 4}
b: + shake                 42/50      44/50   1.0 cm +0.94 / +0.86   {'no_grasp': 6, 'wrong_target': 1, 'dropped': 1}
c: + bias                  26/50      33/50   2.4 cm +1.06 / +0.86   {'wrong_target': 1, 'no_grasp': 22, 'dropped': 1}
d: + bias + shake          35/50      33/50   1.5 cm +1.03 / +0.82   {'no_grasp': 14, 'dropped': 1}


### step 05.3 — does the policy turn the wrist? (`experiments/verne/scripts/wrist_check.py`, seed 0, n_exec 8)

At the first close: how far the fingers are off square, and how the wrist turn follows the cube's angle (expert: slope 1; never turning: off square = the cube's own 21.2°).

```-
                      off square at close (succ / fail)   turn vs cube angle
a: square             15.2°  (10.7 / 20.4)                slope +0.18, corr +0.21
c: + bias             11.5°  (10.1 / 13.1)                slope +0.41, corr +0.63
```

Arms b and d not measured. Films (`scratch/verne_films/`, `05_square_*`): a's `wrong_target` episodes turn the wrong way (cube −30°, wrist +19°) and grip by the corners; c's misses close nearly square but beside the cube.

### wrap-up — squaring up is a data problem, and the shake carries it

**Buys:** with the full shake (b), seed 0 scores 42 / 44 of 50, the best seed of the series (step 04: 36 / 39), with the tightest close (1.0 cm). Squaring demos let the fingers meet the cube square, and the shake shows the policy how to get back to the demo path when it isn't on it.

**Where it fails:**
- Clean squaring demos (a) make things worse (28 / 25): the expert closes within 2° every time, so a crooked grip is a state no demo shows, and the policy hangs there (ep 1044: cube up, held crooked, never carried). The turn is barely learned (slope 0.18), and near ±45° the demos contradict themselves (+44° and −44° look alike, the expert turns opposite ways).
- The aim bias (c) teaches the turn (slope 0.41, corr 0.63) and carrying on from crooked grips (wrong_target 4 → 1), but the aim goes off (miss 2.4 cm): labels aim at the cube's spawn position, latched at reset, while crooked fingers nudge it before the close.
- Bias + shake (d): 35 / 33; its expert placed only 72% of attempts, so its demos are filtered hard. Its misses hover square at the cube and never close (ep 1020): in its demos a close never follows a square wrist.

**Not answered:** one seed per arm; 1 to 2 episodes flip between sweep and replay on the same GPU. No expert ever retries a missed grasp, so no arm teaches it: that needs labels on the policy's own states (DAgger, next).